# `Qwen3.6-35B-A6B-LoopHSSD` (v3.1) · Notebook 1: Zero-Training Diagnostic & Calibration Probes

**Dedicated Hardware, Subspace Alignment, Expert Headroom, `HSSD v3.1` Storage/Throughput Audit, & `1M` Positional Calibration Suite (Colab G4 · `RTX PRO 6000 Blackwell` `96 GiB` · `230 GB` SSD Safe)**

| Specialized Notebook | Purpose |
| :--- | :--- |
| **[`colab_probe.ipynb`](colab_probe.ipynb)** *(you are here)* | **`Gate -1` + `Gate 0a` + `Gate 0p-1..4` + `Gate 1-Ablate` Matrix + Core-Range Sweep**: Zero-training diagnostics, structured `probe_report.json` + `probe_report.md`, calibrated CLI/YAML overrides |
| [`colab_train_e2e.ipynb`](colab_train_e2e.ipynb) | **End-to-End `Loop-HSSD v3.1` Training & Whole-Model Pack Release**: Resumable `Gate 0b -> Gate 4` pipeline + low-disk streaming `safetensors` Whole Model Pack push to HF Hub |
| [`colab_infer_eval.ipynb`](colab_infer_eval.ipynb) | **Shipped Whole-Model On-Weight Inference & Frontier `R=1/2/3` Eval**: `OptimizedLoopInferenceEngine` (`40L/64L/88L A6B`), `1M FP8` + `10M INT4` offload profiler, frontier depth-curve suite |
| [`colab_train.ipynb`](colab_train.ipynb) | Baseline `64L` Track-A (`GDN-2` + `mHC`) single-notebook pipeline |

### Quantitative Go / No-Go Gates Executed Here
1. **`Gate -1` (Double-Precision Math Proofs)**: Verifies `fp64` architectural invariants (`test_loop_hssd_v3_math.py`: disjoint `[513, 576]` horizon, `PeakBoltzmannGate`, post-rotation 2D phase WY `chunk_hssd`, multi-page routed chunk-start root-surprise `s = g^2`, `H_64` ReRoPE commutativity, step-0 `torch.equal` + step-1 non-zero grad).
2. **`Gate 0a` (Hardware Backward & Precision Audit)**: Measures real `tok/s`, peak HBM (`FP8` / `NVFP4` / `BF16` frozen MoE), finite `dL/dx` across all 40 layers, and bit-exact step-0 surgery identity.
3. **`Gate 0p-1..4` (Zero-Training Architectural & `HSSD v3.1` Hardware Calibration)**:
   - **`0p-1` (`probe_expert_headroom`)**: Hard-token NLL headroom under Top-8 router logit perturbations (`>= 0.08` nats target).
   - **`0p-2` (`probe_reentry_anchors`)**: Layer `31 -> 8` cosine alignment, rank-256 Procrustes subspace variance explained, and Pass-2 routing Jaccard across anchor weights `(0.99, 0.90, 0.85, 0.70, 0.50)`.
   - **`0p-3` (`probe_positional_extrapolation`)**: `native` vs `yarn (factor=4.0)` vs `rerope` NLL & passkey needle logit margin shootout.
   - **`0p-4` (`profile_hssd_layer`) + `Gate 1-Ablate` (`hssd_ablation_matrix`)**: Audits Qwen-dim `HSSD v3.1` adapter params (`theta_low_rank=True`, `~1.95M/layer`), unique backing-storage bytes (`81-89 MiB/layer` in `state_dtype="bfloat16"`, zero prompt-length storage leak), prefill / streamed-prefill / `k=2` decode / fwd+bwd throughput vs single-state `GDN`, and the 6-way matched-budget mechanism-isolation matrix.
4. **Verbose Crash Forensics (`qloop.debug`)**: Every subprocess writes `<run_dir>/crash/crash_<ts>.md` (`Hints`, `Traceback`, per-frame tensor stats, CUDA memory). Set `DEBUG_LEVEL = 2` (NaN/Inf module hooks + C++ stacktraces) or `3` (`CUDA_LAUNCH_BLOCKING=1` + autograd anomaly + memory snapshot).

In [ ]:
REPO = "https://github.com/BaiTian6641/Qwen3.6-35B-A3B-Loop60.git"
BRANCH = "main"
%cd /content
!test -d /content/qloop/.git && test ! -d /content/qloop_repo && mv /content/qloop /content/qloop_repo || true
!test -d /content/qloop_repo || git clone -b $BRANCH $REPO /content/qloop_repo
%cd /content/qloop_repo
!git pull --ff-only
!bash scripts/colab_setup.sh

In [ ]:
import importlib
import os
import sys

if os.path.abspath("src") not in sys.path:
    sys.path.insert(0, os.path.abspath("src"))
if "qloop" in sys.modules and getattr(sys.modules["qloop"], "__file__", None) is None:
    for _k in [k for k in sys.modules if k == "qloop" or k.startswith("qloop.")]:
        del sys.modules[_k]
importlib.invalidate_caches()

from qloop.colab import hub_status, q, reclaim_disk, resolve_base_model_name, run_probe_notebook_suite, setup_env

BASE_PRECISION = "fp8"  # "fp8" (Qwen/Qwen3.6-35B-A3B-FP8, ~35 GiB) | "nvfp4" (unsloth/Qwen3.6-35B-A3B-NVFP4-Fast, ~21 GiB) | "bf16"
HF_REPO = "TassP0lar/Qwen3.6-35B-A3B-Loop64"  # target sync repo for artifacts/s0_probe/*
SMOKE_TOY = False                             # True = fast CPU/GPU smoke test on ToyForCausalLM
PUSH_PROBE_TO_HUB = True                      # True = commit probe_report.{json,md} to HF_REPO via HF_TOKEN
DEBUG_LEVEL = 1                               # 1=normal | 2=NaN/Inf module hooks | 3=CUDA_LAUNCH_BLOCKING

setup_env(
    hf_repo=HF_REPO,
    run_root="/content/runs",
    hf_home="/content/hf_cache",
    require_hf_token=PUSH_PROBE_TO_HUB,
    base_precision=BASE_PRECISION,
    auto_install_runtime_deps=not SMOKE_TOY,
)
doctor_cmd = "doctor --freeze requirements/colab.lock.txt"
if not SMOKE_TOY:
    doctor_cmd += f" --model {resolve_base_model_name(BASE_PRECISION)}"
q(f"{doctor_cmd} --debug {DEBUG_LEVEL}")
if PUSH_PROBE_TO_HUB:
    hub_status(HF_REPO)

### 1 · Execute Complete Zero-Training Probe Suite (`Gate -1`, `Gate 0a`, `Gate 0p-1..4`, Core-Range Sweep)
Runs all mathematical, backward-precision, subspace-alignment, expert-headroom, `1M` positional, and `HSSD v3.1` single-layer storage/throughput probes, writes `probe_report.json` + `probe_report.md`, renders the structured calibration report inline, and prints calibrated overrides for `colab_train_e2e.ipynb`.

In [ ]:
PROBE_SEQ_LEN = 256 if SMOKE_TOY else 2048
PROBE_N_SEQS = 2 if SMOKE_TOY else 4
TARGET_VIRTUAL_LAYERS = 24 if SMOKE_TOY else 88

probe_report = run_probe_notebook_suite(
    config_path="configs/stages/s0_probe.yaml",
    hf_repo=HF_REPO,
    base_precision=BASE_PRECISION,
    smoke_toy=SMOKE_TOY,
    seq_len=PROBE_SEQ_LEN,
    n_seqs=PROBE_N_SEQS,
    target_virtual=TARGET_VIRTUAL_LAYERS,
    run_math_gate=True,
    push_to_hub=PUSH_PROBE_TO_HUB,
    debug=DEBUG_LEVEL,
)

md_text = probe_report.get("markdown", "")
if md_text:
    try:
        from IPython.display import Markdown, display

        display(Markdown(md_text))
    except Exception:
        print(md_text)

print("\n=== Calibrated Recommended Overrides (pass via `-s` or lock in stage YAML) ===")
for k, v in sorted((probe_report.get("recommended_overrides") or {}).items()):
    print(f"  -s {k}={v}")
print("\nSaved artifacts:", probe_report.get("json_path"), "|", probe_report.get("md_path"))

### 2 · `Gate 0p-4` & `Gate 1-Ablate`: `HSSD v3.1` Backing-Storage Audit, Throughput vs GDN Baseline, & 6-Way Ablation Matrix
Inspects unique backing-storage bytes (`HSSDCacheState.unique_storage_bytes()`, verifying zero prompt-length storage retention across `page_states`, `boundary_ring`, `swa_{k,v}_ring`, and `float64` `theta_cum`), compares full/streamed prefill, `k=2` single-token decode, and `fwd+bwd` training throughput against the single-state `GDN` baseline, and surfaces the 6-way matched-budget mechanism-isolation matrix (`Gate 1-Ablate`).

In [ ]:
import json

hssd_prof = probe_report.get("hssd_layer_profile") or {}
ablation_mat = probe_report.get("hssd_ablation_matrix") or {}
rcache = hssd_prof.get("retained_cache") or {}
dims = hssd_prof.get("dimensions") or {}

u_bytes = rcache.get("unique_storage_bytes", rcache.get("storage_bytes", 0))
u_mib = float(u_bytes) / (1024.0 * 1024.0) if u_bytes else float(rcache.get("unique_storage_mib", 0.0))
compact_ok = rcache.get("has_compact_storage", rcache.get("compact_storage", True))

print("=== Gate 0p-4: Single-Layer HSSD v3.1 Parameter & Backing-Storage Audit ===")
print("Dimensions             :", dims)
print(
    f"Adapter Params         : {int(hssd_prof.get('adapter_params_per_layer') or 0):,} / layer  |  "
    f"{int(hssd_prof.get('adapter_params_30_layers') or 0):,} / 30L  "
    f"(theta share: {float(hssd_prof.get('theta_param_fraction') or 0.0) * 100.0:.1f}%)"
)
print(
    f"Retained Decode Cache  : {int(u_bytes):,} unique bytes ({u_mib:.2f} MiB / slot, "
    f"has_compact_storage={compact_ok})"
)
print(
    f"Multi-Pass Projections : R=1 (30 slots)={30.0 * u_mib / 1024.0:.2f} GiB  |  "
    f"R=2 (48 slots)={48.0 * u_mib / 1024.0:.2f} GiB  |  "
    f"R=3 (66 slots)={66.0 * u_mib / 1024.0:.2f} GiB"
)

print("\n=== Prefill / Streamed-Prefill / Decode / Fwd+Bwd vs GDN Baseline ===")
for bench_key in (
    "full_prefill",
    "streamed_prefill",
    "single_token_decode",
    "training_fwd_bwd",
    "gdn_baseline",
):
    print(f"  {bench_key:20s}: {json.dumps(hssd_prof.get(bench_key) or {}, default=str)}")

print("\n=== Gate 1-Ablate: 6-Way Matched-Budget HSSD Mechanism Isolation Matrix ===")
for variant_id, variant_cfg in ablation_mat.items():
    print(f"  [{variant_id}] -> {json.dumps(variant_cfg, default=str)}")

### 3 · Structured JSON Diagnostics & Optional Precision / ESFT Router Affinity Scan
Inspects the structured `gates` & `summary` JSON payload, optionally compares `FP8` (`~35 GiB`) vs `NVFP4` (`~21 GiB`) backward throughput (`COMPARE_NVFP4_BASE = True`), or runs the `256`-batch ESFT router affinity scan (`RUN_ESFT_SCAN = True`) while reclaiming transient SSD staging space.

In [ ]:
import json

COMPARE_NVFP4_BASE = False  # True = also probe unsloth/Qwen3.6-35B-A3B-NVFP4-Fast backward & tok/s
RUN_ESFT_SCAN = False       # True = run 256-batch ESFT router affinity scan -> artifacts/s0_probe/esft_stats.json

print("=== Structured Probe Summary & Gate Verdicts ===")
print(
    json.dumps(
        {
            "summary": probe_report.get("summary", {}),
            "gates": probe_report.get("gates", {}),
            "recommended_overrides": probe_report.get("recommended_overrides", {}),
        },
        indent=2,
        default=str,
    )
)

if COMPARE_NVFP4_BASE and not SMOKE_TOY:
    q(
        f"probe -c configs/stages/s0_probe.yaml --backward --seq-len {PROBE_SEQ_LEN} "
        f"--debug {DEBUG_LEVEL} -s model.name=unsloth/Qwen3.6-35B-A3B-NVFP4-Fast "
        f"-s checkpoint.sync.hf_repo={HF_REPO}",
        check=False,
    )

if RUN_ESFT_SCAN:
    model_ov = "-s model.name=toy" if SMOKE_TOY else f"-s model.name={resolve_base_model_name(BASE_PRECISION)}"
    batches = 8 if SMOKE_TOY else 256
    q(
        f"scan-experts -c configs/stages/s0_probe.yaml --batches {batches} --seq-len {PROBE_SEQ_LEN} "
        f"--out /content/runs/s0_probe/esft_stats.json --debug {DEBUG_LEVEL} "
        f"{model_ov} -s checkpoint.sync.hf_repo={HF_REPO}"
    )
    reclaim_disk(keep="s0_probe")